# 🪞 FACE LAB — 임베딩 공간과 각도 마진 (ArcFace의 기하학)

> **[26년 3기] NPU 활용 온디바이스 AI 프로그래밍** · 특별 세션 · **CPU 런타임으로 충분**

---

## 🔑 핵심 메시지

> **"안면 인식은 분류가 아니라 비교다 — 그리고 비교의 품질은 각도가 결정한다."**
>
> 분류기는 "이 사람은 누구?"에 답하지만, 실전 안면 인식은 **"이 두 사진이 같은 사람인가?"** 에 답해야 합니다.
> 훈련 때 본 적 없는 사람도요. 그래서 모델은 라벨이 아니라 **임베딩(방향 벡터)** 을 출력하고,
> 판정은 두 벡터의 **각도** 하나로 끝납니다. 오늘 우리는 그 각도 공간을 직접 학습시키고,
> 마진 하나(cos(θ+m))가 공간의 기하를 어떻게 조이는지 숫자로 확인합니다.

## 📋 실습 로드맵

| Part | 주제 | 도구 | 재현성 |
|---|---|---|---|
| 1 | 진짜 얼굴 → 512차원 벡터 | facenet-pytorch | 📊 |
| 2 | 코사인 유사도 해부 | 순수 numpy | ✅ |
| 3 | 검증 문제와 FAR/FRR — "99%"의 거짓말 | seeded numpy 학습 | ✅ EER 8.78% |
| 4 | ArcFace 각도 마진 — 공간을 조이다 | 〃 | ✅ EER 8.03% |
| 5 | 내 얼굴에 임계값 적용 | facenet + Part 3 논리 | 📊 |
| 6 | 리포트 + 윤리 | — | — |

## ⚖️ 시작 전 — 윤리 원칙 (이 랩의 규칙)
1. **본인과 동의한 지인의 사진만** 사용합니다 (타인·유명인 사진 금지)
2. 사진은 코랩 세션 메모리에서만 처리되고 세션 종료 시 사라집니다 — 그래도 민감하면 업로드하지 마세요 (digits 파트만으로 전 과정 이수 가능)
3. 안면 인식 기술의 오용(무단 감시, 동의 없는 식별)은 이 랩이 가르치는 것의 반대편입니다 — Part 6에서 정면으로 다룹니다


---
# Part 0 · 환경 설정

In [ ]:
# [0-1] 설치 — 실제 얼굴 파트용 facenet-pytorch (사전학습 InceptionResnetV1 + MTCNN)
!pip install -q facenet-pytorch
import numpy as np
import matplotlib.pyplot as plt
import torch
print("✅ 준비 완료 (CPU로 충분)")

---
# Part 1 · 진짜 얼굴 → 512차원 벡터

파이프라인: **MTCNN**(얼굴 찾기·정렬) → **InceptionResnetV1**(VGGFace2 사전학습) → **512차원 임베딩**.

준비물: **본인 사진 2장**(다른 날/각도) + **동의받은 지인 사진 1장**. 파일명을 `me1.jpg, me2.jpg, friend.jpg`로 맞춰 업로드하세요.

In [ ]:
# [1-1] 사진 업로드 → 얼굴 정렬 → 임베딩 추출
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
from google.colab import files

up = files.upload()          # me1.jpg, me2.jpg, friend.jpg
names = ["me1", "me2", "friend"]

mtcnn = MTCNN(image_size=160, margin=14)
resnet = InceptionResnetV1(pretrained='vggface2').eval()

embs, faces = {}, {}
for n in names:
    img = Image.open(f"{n}.jpg").convert("RGB")
    face = mtcnn(img)                       # 정렬된 160×160 얼굴 텐서
    assert face is not None, f"{n}: 얼굴 미검출 — 밝은 정면 사진으로 재시도"
    with torch.no_grad():
        e = resnet(face.unsqueeze(0))[0]
    embs[n] = (e / e.norm()).numpy()        # 단위 벡터로 정규화
    faces[n] = face

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, n in zip(axes, names):
    ax.imshow((faces[n].permute(1,2,0).numpy()*0.5+0.5).clip(0,1))
    ax.set_title(n); ax.axis('off')
plt.suptitle("MTCNN이 정렬한 얼굴 (모델이 실제로 보는 입력)"); plt.show()
print(f"임베딩 차원: {embs['me1'].shape}   # ✅ (512,)")

In [ ]:
# [1-2] 코사인 유사도 행렬 — 첫 대면
sim = np.zeros((3,3))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        sim[i,j] = float(embs[a] @ embs[b])

print("       " + "  ".join(f"{n:>7}" for n in names))
for i, a in enumerate(names):
    print(f"{a:>7}" + "  ".join(f"{sim[i,j]:7.3f}" for j in range(3)))
print()
print("📊 기대 패턴: me1↔me2 (본인쌍) > me↔friend (타인쌍)")
print("   본인쌍은 대략 0.6~0.9, 타인쌍은 대략 0.0~0.4가 일반적 — 여러분 값은?")

> 🧑‍🏫 **강사 노트 (양방향)**: 본인쌍 유사도가 타인쌍보다 **낮게 나오는 학생이 간혹 있습니다** — 대부분 (a) 한 장이 심한 측면/저조도, (b) MTCNN이 배경 얼굴을 잡은 경우입니다. [1-1]의 정렬 얼굴 그림으로 원인을 진단하세요 (HAND LAB의 진단 습관: 임계값 문제인가 입력 문제인가). 정상 사례가 대부분이면 "그런데 0.62면 같은 사람인가요? 0.55면요?"라고 물어 Part 3의 임계값 문제로 넘어가는 것이 최고의 전환입니다.

---
# Part 2 · 코사인 유사도 해부 — 판정의 전부는 각도

$$\cos\theta = \frac{a \cdot b}{|a||b|}$$

단위 벡터끼리는 그냥 **내적 = 코사인**. POSE/HAND LAB의 `joint_angle`과 같은 수학의 세 번째 등장입니다 — 이번엔 관절이 아니라 512차원 정체성 벡터 사이의 각도.

In [ ]:
# [2-1] 손계산 검증 4종 — 완전 결정적 ✅
def cos_sim(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"같은 방향  [1,0]·[1,0]   : {cos_sim([1,0],[1,0]):7.4f}   (θ=0°)")
print(f"직교       [1,0]·[0,1]   : {cos_sim([1,0],[0,1]):7.4f}   (θ=90°)")
print(f"정반대     [1,0]·[-1,0]  : {cos_sim([1,0],[-1,0]):7.4f}   (θ=180°)")
print(f"손계산용   [3,4]·[4,3]   : {cos_sim([3,4],[4,3]):7.4f}   (=24/25, θ≈16.26°)")

**✅ 기대 출력** — 마지막 줄 손계산: $(3·4+4·3)/(5·5) = 24/25 = 0.9600$. 종이로 확인하세요.

핵심 통찰: 임베딩을 단위 벡터로 정규화하면 **크기 정보가 사라지고 방향만 남습니다** — 조명·해상도 같은 강도 요인을 버리고 정체성(방향)만 비교하겠다는 설계 선언입니다. HAND LAB의 정규화(위치·크기 소거)와 같은 철학, 다른 공간.

---
# Part 3 · 검증 문제와 FAR/FRR — "인식률 99%"의 거짓말

실전 질문은 분류("누구?")가 아니라 **검증**("같은 사람?")입니다. 라벨링된 얼굴 데이터셋 없이 이 문제를 결정적으로 실험하기 위해, **digits(0~9 숫자)를 10명의 '신원'으로** 간주합니다 — 같은 숫자 쌍 = 본인쌍(genuine), 다른 숫자 쌍 = 타인쌍(impostor). (다운로드 불필요, seeded, ✅ 완전 재현)

먼저 **일반 softmax**로 2차원 임베딩을 학습합니다 — 2차원인 이유: 단위원 위에 그대로 그릴 수 있어서, Part 4의 기하 대비가 눈에 보입니다.

In [ ]:
# [3-1] 데이터 + 학습 함수 (softmax / arcface 공용) — 순수 numpy 역전파
from sklearn.datasets import load_digits
X, y = load_digits(return_X_y=True); X = X / 16.0
perm = np.random.default_rng(7).permutation(len(X))
tr, te = perm[:1200], perm[1200:]
print(f"'신원' 10명, 학습 {len(tr)} / 평가 {len(te)}")

def train(mode, m=0.25, s=16.0, epochs=1500, lr=0.5, seed=0, warmup=300):
    """64→48 tanh→2 임베딩 → 단위화 → 클래스 프로토타입과 cos → CE
       mode='arcface'면 warmup 이후 타깃 로짓에 각도 마진: cos(θ+m)"""
    rs = np.random.default_rng(seed)
    W1 = rs.normal(0,.3,(64,48)); b1 = np.zeros(48)
    W2 = rs.normal(0,.3,(48,2));  b2 = np.zeros(2)
    Wc = rs.normal(0,.3,(2,10))               # 클래스 프로토타입(방향)
    Xtr, Ytr, yv = X[tr], np.eye(10)[y[tr]], y[tr]
    N = len(Xtr); ar = np.arange(N)
    for ep in range(epochs):
        use_m = (mode == "arcface" and ep >= warmup)
        H = np.tanh(Xtr@W1+b1); Z = H@W2+b2
        nz = np.linalg.norm(Z,axis=1,keepdims=True)+1e-9; Zn = Z/nz
        nw = np.linalg.norm(Wc,axis=0,keepdims=True)+1e-9; Wn = Wc/nw
        cos = Zn @ Wn
        logits = s * cos
        if use_m:                              # ── ArcFace 핵심 한 줄 ──
            c = np.clip(cos[ar,yv], -1+1e-7, 1-1e-7)
            logits = logits.copy()
            logits[ar,yv] = s*(c*np.cos(m) - np.sqrt(1-c**2)*np.sin(m))  # cos(θ+m)
        logits -= logits.max(1,keepdims=True)
        P = np.exp(logits); P /= P.sum(1,keepdims=True)
        G = (P - Ytr)/N; dcos = s*G
        if use_m:
            c = np.clip(cos[ar,yv], -1+1e-7, 1-1e-7)
            dcos[ar,yv] *= (np.cos(m) + c*np.sin(m)/np.sqrt(1-c**2))
        dZn = dcos@Wn.T; dWn = Zn.T@dcos       # 정규화 통과 역전파
        dZ  = (dZn - Zn*np.sum(dZn*Zn,axis=1,keepdims=True))/nz
        dWc = (dWn - Wn*np.sum(dWn*Wn,axis=0,keepdims=True))/nw
        dW2 = H.T@dZ; db2 = dZ.sum(0)
        GH = (dZ@W2.T)*(1-H**2); dW1 = Xtr.T@GH; db1 = GH.sum(0)
        W1-=lr*dW1; b1-=lr*db1; W2-=lr*dW2; b2-=lr*db2; Wc-=lr*dWc
    def embed(Xin):
        Z = np.tanh(Xin@W1+b1)@W2+b2
        return Z/(np.linalg.norm(Z,axis=1,keepdims=True)+1e-9)
    Wn = Wc/(np.linalg.norm(Wc,axis=0,keepdims=True)+1e-9)
    acc = float(np.mean((embed(X[te])@Wn).argmax(1)==y[te]))
    return acc, embed

In [ ]:
# [3-2] softmax 임베딩 학습 (📊 코랩 CPU 약 1~2분) + 단위원 시각화
acc_s, emb_s = train("softmax")
print(f"softmax 분류 정확도: {acc_s*100:.2f}%")

Zs, yte = emb_s(X[te]), y[te]
fig, ax = plt.subplots(figsize=(6.5,6.5))
th = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(th), np.sin(th), color='gray', lw=0.8, alpha=.5)
for c in range(10):
    z = Zs[yte==c]
    ax.scatter(z[:,0], z[:,1], s=10, label=str(c), alpha=.7)
ax.set_aspect('equal'); ax.legend(ncol=5, fontsize=8, loc='upper center',
                                  bbox_to_anchor=(0.5,-0.02))
ax.set_title("softmax 임베딩 — 단위원 위 10개 '신원'"); ax.axis('off')
plt.show()
print("📊 관찰: 군집이 나뉘긴 했지만 경계가 서로 닿아 있습니다 — 이게 문제의 씨앗")

In [ ]:
# [3-3] 본인쌍/타인쌍 유사도 분포 → 임계값 스윕 → FAR/FRR/EER
def pairs_sims(embed, n=3000, seed=42):
    rng = np.random.default_rng(seed)
    Zte, yy = embed(X[te]), y[te]
    by = {c: np.where(yy==c)[0] for c in range(10)}
    gen, imp = [], []
    while len(gen) < n:
        c = rng.integers(10); i, j = rng.choice(by[c], 2, replace=False)
        gen.append(float(Zte[i] @ Zte[j]))
    while len(imp) < n:
        c1, c2 = rng.choice(10, 2, replace=False)
        imp.append(float(Zte[rng.choice(by[c1])] @ Zte[rng.choice(by[c2])]))
    return np.array(gen), np.array(imp)

def far_frr_eer(gen, imp):
    thrs = np.linspace(-1, 1, 2001)
    far = np.array([(imp >= t).mean() for t in thrs])   # 타인을 수락한 비율
    frr = np.array([(gen <  t).mean() for t in thrs])   # 본인을 거부한 비율
    k = np.argmin(np.abs(far - frr))
    return thrs, far, frr, float((far[k]+frr[k])/2*100), float(thrs[k])

gen_s, imp_s = pairs_sims(emb_s)
thrs, far, frr, eer_s, thr_s = far_frr_eer(gen_s, imp_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(imp_s, bins=36, range=(-1,1), alpha=.6, color='#ff5d5d', label='타인쌍')
axes[0].hist(gen_s, bins=36, range=(-1,1), alpha=.6, color='#39d98a', label='본인쌍')
axes[0].axvline(thr_s, color='#4dc9ff', lw=2, label=f'EER 임계값 {thr_s:.3f}')
axes[0].legend(); axes[0].set_title("두 분포는 반드시 겹친다"); axes[0].set_xlabel("코사인 유사도")
axes[1].plot(thrs, far*100, color='#ff5d5d', label='FAR (타인 수락)')
axes[1].plot(thrs, frr*100, color='#39d98a', label='FRR (본인 거부)')
axes[1].axvline(thr_s, color='#4dc9ff', lw=1.5)
axes[1].set_ylim(0, 40); axes[1].legend(); axes[1].set_xlabel("임계값")
axes[1].set_title(f"트레이드오프 — 교차점이 EER = {eer_s:.2f}%")
plt.tight_layout(); plt.show()

print(f"softmax 검증 성능: EER {eer_s:.2f}% @ threshold {thr_s:.3f}")

**✅ 기대 출력** (seed 고정)
```
softmax 분류 정확도: 88.11%
softmax 검증 성능: EER 8.78% @ threshold 0.835
```

### 📌 "인식률 99%"가 왜 무의미한가

두 분포가 겹치는 한, 임계값을 어디에 두든 **FAR과 FRR 중 하나는 반드시 손해**입니다:
- 임계값↑ → 보안↑ (FAR↓) 그러나 본인이 자주 거부됨 (FRR↑) — 도어락이 주인을 안 열어줌
- 임계값↓ → 편의↑ (FRR↓) 그러나 타인이 뚫음 (FAR↑) — 보안 사고

"인식률" 한 숫자는 이 트레이드오프를 숨깁니다. 정직한 성능 표기는 **"FAR 0.1%에서 FRR 몇 %"** 처럼 운영점을 명시하는 것 — 우리 보드의 confidence 50% 필터를 정할 때 했던 고민과 정확히 같은 구조입니다. 근본 해법은 하나뿐: **두 분포 자체를 떨어뜨리는 것.** 그래서 Part 4입니다.

---
# Part 4 · ArcFace 각도 마진 — 공간을 조이다 🎬

아이디어는 훈련 규칙 한 줄입니다: 정답 클래스의 로짓을 $\cos\theta$ 대신 $\cos(\theta + m)$ 으로 계산.

> "네 각도에 페널티 0.25rad(≈14°)을 더해도 이길 만큼 **프로토타입에 바짝 붙어라**"

같은 데이터·같은 seed·같은 구조 — 바뀌는 건 이 한 줄뿐입니다 ([3-1]의 `use_m` 분기).

In [ ]:
# [4-1] ArcFace 학습 + 두 공간 나란히 보기
acc_a, emb_a = train("arcface", m=0.25)
print(f"arcface 분류 정확도: {acc_a*100:.2f}%   (softmax {acc_s*100:.2f}%)")

Za = emb_a(X[te])
fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
for ax, Z, name in [(axes[0], Zs, "softmax"), (axes[1], Za, f"arcface (m=0.25)")]:
    ax.plot(np.cos(th), np.sin(th), color='gray', lw=0.8, alpha=.5)
    for c in range(10):
        z = Z[yte==c]
        ax.scatter(z[:,0], z[:,1], s=10, alpha=.7)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(name)
plt.suptitle("같은 데이터, 같은 seed — 마진 한 줄의 기하학적 효과", y=0.98)
plt.show()

In [ ]:
# [4-2] 정량 비교 — 퍼짐, EER, 정확도 최종 스코어보드
def intra_spread(Z, yy):
    stds = []
    for c in range(10):
        z = Z[yy==c]; ang = np.arctan2(z[:,1], z[:,0])
        mean = np.arctan2(np.sin(ang).mean(), np.cos(ang).mean())
        d = np.angle(np.exp(1j*(ang-mean)))
        stds.append(np.degrees(np.sqrt((d**2).mean())))
    return float(np.mean(stds))

gen_a, imp_a = pairs_sims(emb_a)
_, _, _, eer_a, thr_a = far_frr_eer(gen_a, imp_a)
sp_s, sp_a = intra_spread(Zs, yte), intra_spread(Za, yte)

print("┌────────────────────────────────────────────────┐")
print(f"│                    softmax      arcface        │")
print(f"│ 분류 정확도        {acc_s*100:6.2f}%      {acc_a*100:6.2f}%        │")
print(f"│ 검증 EER           {eer_s:6.2f}%      {eer_a:6.2f}%        │")
print(f"│ 클래스 내 퍼짐     {sp_s:6.1f}°      {sp_a:6.1f}°        │")
print("└────────────────────────────────────────────────┘")

**✅ 기대 출력** (seed 고정 — 정확히 이 값)
```
│                    softmax      arcface        │
│ 분류 정확도         88.11%       88.94%        │
│ 검증 EER             8.78%        8.03%        │
│ 클래스 내 퍼짐       16.8°        14.3°        │
```

### 🎬 클라이맥스 정리

마진 한 줄이 만든 변화 — 군집이 **16.8° → 14.3°** 로 조여지고, 그 결과 본인쌍/타인쌍 분포가 벌어져 **EER 8.78% → 8.03%**. 분류 정확도가 아니라 **공간의 기하**를 최적화했기 때문에, 훈련에서 본 적 없는 쌍의 비교(검증)가 좋아진 것입니다.

> 정직한 노트: 개선 폭은 극적이기보다 **일관적**입니다 — 2차원 병목(원 위 10클래스 = 클래스당 36°)이라는 좁은 무대라서요. 512차원의 실제 ArcFace는 훨씬 여유로운 공간에서 같은 원리로 작동합니다 (리포트 ②에서 차원을 늘려 직접 확인). 또한 m을 키우면 정확도와 EER이 **서로 다른 방향**으로 움직일 수 있습니다 — 리포트 ①의 주제.

---
# Part 5 · 내 얼굴에 임계값 적용 — 이론이 도어락이 되는 순간

Part 1의 유사도에 Part 3의 임계값 논리를 적용합니다. 여러분이 방금 만든 것은 사실상 **얼굴 잠금 해제의 판정부**입니다.

In [ ]:
# [5-1] 미니 얼굴 검증기
THRESHOLD = 0.55        # 🔁 바꿔가며 실험 — Part 3에서 배운 그 트레이드오프

print(f"임계값 {THRESHOLD} 기준 판정:")
for a, b in [("me1","me2"), ("me1","friend"), ("me2","friend")]:
    s_ab = float(embs[a] @ embs[b])
    verdict = "✅ 같은 사람" if s_ab >= THRESHOLD else "❌ 다른 사람"
    print(f"  {a} ↔ {b}: {s_ab:.3f} → {verdict}")
print()
print("📊 실험: THRESHOLD를 0.3 / 0.55 / 0.8로 바꿔보세요.")
print("   0.8에서 본인쌍이 거부되면 그것이 FRR, 0.3에서 타인쌍이 통과하면 그것이 FAR —")
print("   Part 3의 그래프가 여러분 얼굴에서 재현되는 것입니다.")

---
# Part 6 · 리포트 과제 (3종 중 2종) + 윤리

### 실험 ① — 마진의 양날: m 스윕
`m`을 0.15 / 0.25 / 0.30 / 0.40으로 바꿔 분류 정확도·EER·퍼짐을 표로 기록하세요.
- 예고: **정확도가 가장 좋은 m과 EER이 가장 좋은 m이 다를 수 있습니다** (우리 사전 실험에서 m=0.30이 정확도 90.12%로 최고였지만 EER은 m=0.25가 우세했습니다). 두 지표가 갈리는 이유를 "무엇을 최적화하는가" 관점에서 논하세요.

### 실험 ② — 차원의 해방
임베딩 차원을 2 → 8 → 32로 바꿔 (W2와 Wc 크기 수정) 같은 비교를 반복하세요. 2차원 병목이 풀리면 softmax와 arcface의 격차는 어떻게 변하나요? "마진은 언제 가장 필요한가"에 답하세요.

### 실험 ③ — 내 얼굴 스트레스 테스트
본인 사진을 5장으로 늘려(정면/측면/저조도/안경/모자) 5×5 유사도 행렬을 만들고, 어떤 변화가 유사도를 가장 크게 떨어뜨리는지 순위를 매기세요. "임베딩이 불변이어야 할 것과 민감해야 할 것"의 목록을 만들어 보세요.

---

## ⚖️ 윤리 — 이 기술의 무게

오늘 만든 검증기의 원리는 실제 제품과 같습니다. 그래서 다음을 기억해야 합니다:
- **FAR/FRR은 사람마다 다릅니다** — 학습 데이터에 적게 등장한 인구집단에서 오류율이 높아지는 편향이 보고되어 왔습니다. "평균 EER"은 개인의 경험을 대변하지 못합니다.
- **동의가 기본값입니다** — 검증(내 기기의 내 얼굴)과 식별(모르는 군중에서 누군가 찾기)은 기술적으로 이웃이지만 윤리적으로는 다른 세계입니다.
- 여러분이 이 기술을 배포하는 입장이 된다면: 운영점(FAR/FRR) 공개, 집단별 성능 검증, 옵트인 동의가 최소한의 기준입니다.

## ✅ 체크포인트 — 오늘 확보해야 할 3가지
1. **유사도 행렬** [1-2] — 본인쌍 > 타인쌍 패턴
2. **FAR/FRR 그래프** [3-3] — EER 8.78% 재현
3. **최종 스코어보드** [4-2] — 8.78→8.03%, 16.8°→14.3°

> 🔗 **NPU 파이프라인과의 연결 (교육자 노트)**
> 스마트폰 얼굴 잠금 해제가 이 랩의 실물입니다 — 그리고 그 구조가 온디바이스 AI의 모범 답안입니다: **임베딩 추출은 기기 안 NPU에서**(수 ms, 저전력), **내 얼굴 템플릿은 기기 밖으로 나가지 않고**, 판정은 코사인 내적 하나(사실상 공짜). 서버가 필요 없는 이유는 비교 연산이 512차원 내적일 뿐이기 때문 — "모델은 가속기에, 로직은 호스트에"의 가장 아름다운 사례이자, **엣지 = 프라이버시**라는 등식이 성립하는 지점입니다. 우리 보드로 치면: 임베딩 네트워크를 PTQ로 INT8화해 NPU에 올리고, 템플릿 매칭은 CPU 몇 줄 — YOLO+NMS 분리와 같은 패턴의 신원 버전입니다.